# DP Ph1 RLC State-Space Extraction

This notebook validates MNA state-space extraction for a single-phase dynamic-phasor (DP) RLC circuit.

Three equivalent model variants are considered:

1. **Native DP MNA components**: the circuit is built from standard DP Ph1 MNA components.
2. **SSN load**: the RLC load is replaced by a generic two-terminal V-type DP SSN component.
3. **SSN branch**: the series RL branch is replaced by a generic two-terminal V-type DP SSN component.

For each case, the notebook extracts the discrete-time state matrix, computes modal quantities using `StateSpaceModalAnalysis`, and checks the extracted eigenvalues against the analytical native-frame DP reference.

In [1]:
import numpy as np
import dpsimpy

dp = dpsimpy.dp
ph1 = dp.ph1

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
StateSpaceModalAnalysis = dpsimpy.StateSpaceModalAnalysis

## Parameters

The topology is:

Voltage source -> R_branch -> L_branch -> R_load -> L_load -> C_load -> ground

In [2]:
time_step = 1e-4
final_time = 1e-3
frequency = 50.0

source_voltage = 1.0 * np.exp(1j * 0.0)

branch_resistance = 0.2
branch_inductance = 0.02

load_resistance = 1.0
load_inductance = 0.05
load_capacitance = 100e-6

## Helper functions

In [ ]:
def print_complex_vector(values, indent="  "):
    for idx, value in enumerate(values):
        print(f"{indent}[{idx}] {value}")


def create_rlc_load_ssn_matrices():
    # x = [u_C; i], input = v, output = i.
    a = np.zeros((2, 2))
    a[0, 1] = 1.0 / load_capacitance
    a[1, 0] = -1.0 / load_inductance
    a[1, 1] = -load_resistance / load_inductance

    b = np.zeros((2, 1))
    b[1, 0] = 1.0 / load_inductance

    c = np.zeros((1, 2))
    c[0, 1] = 1.0

    d = np.zeros((1, 1))

    return a, b, c, d


def create_rl_branch_ssn_matrices():
    # x = i, input = v, output = i.
    a = np.zeros((1, 1))
    a[0, 0] = -branch_resistance / branch_inductance

    b = np.zeros((1, 1))
    b[0, 0] = 1.0 / branch_inductance

    c = np.eye(1)
    d = np.zeros((1, 1))

    return a, b, c, d


def map_continuous_to_discrete(lam):
    return (2.0 + time_step * lam) / (2.0 - time_step * lam)


def expected_modes():
    total_resistance = branch_resistance + load_resistance
    total_inductance = branch_inductance + load_inductance

    alpha = total_resistance / (2.0 * total_inductance)
    omega0 = np.sqrt(1.0 / (total_inductance * load_capacitance))
    omega_d = np.sqrt(omega0**2 - alpha**2)
    omega_shift = 2.0 * np.pi * frequency

    lambda_physical = np.array(
        [
            -alpha + 1j * omega_d,
            -alpha - 1j * omega_d,
        ],
        dtype=np.complex128,
    )

    # Native DP / shifted-frequency frame.
    lambda_dp = lambda_physical - 1j * omega_shift

    # The real-valued augmented extraction contains the conjugates as well.
    lambda_dp_augmented = np.array(
        [
            lambda_dp[0],
            lambda_dp[1],
            np.conj(lambda_dp[0]),
            np.conj(lambda_dp[1]),
        ],
        dtype=np.complex128,
    )

    z_dp = map_continuous_to_discrete(lambda_dp)
    z_dp_augmented = map_continuous_to_discrete(lambda_dp_augmented)

    return lambda_physical, lambda_dp, lambda_dp_augmented, z_dp, z_dp_augmented


lambda_physical, lambda_dp, lambda_dp_augmented, z_dp, z_dp_augmented = expected_modes()

print("Physical continuous-time modes before frequency shifting:")
print_complex_vector(lambda_physical)

print("\nNative DP continuous-time modes in the shifted frame:")
print_complex_vector(lambda_dp)

print("\nExpected finite modes in the real-valued augmented extraction:")
print_complex_vector(lambda_dp_augmented)

print("\nExpected trapezoidal discrete-time DP modes:")
print_complex_vector(z_dp)

## Simulation and modal-analysis helpers

In [4]:
def configure_and_run(system, sim_name):
    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.set_domain(Domain.DP)
    sim.set_solver(Solver.MNA)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.do_state_space_extraction(True)
    sim.run()
    return sim


def get_modal_results(sim):
    extractor = sim.get_state_space_extractor()

    modal_analysis = StateSpaceModalAnalysis(extractor)
    modal_analysis.update()

    ad = np.array(extractor.get_discrete_state_matrix(), copy=True)
    z = np.array(modal_analysis.get_discrete_eigenvalues(), copy=True).reshape(-1)
    lambdas = np.array(modal_analysis.get_continuous_eigenvalues(), copy=True).reshape(
        -1
    )

    return {
        "state_count": extractor.get_state_count(),
        "Ad": ad,
        "z": z,
        "lambda": lambdas,
    }


def sort_complex(values):
    return np.array(
        sorted(values, key=lambda value: (round(value.imag, 8), round(value.real, 8)))
    )

## Case 1: native DP Ph1 MNA components

In [5]:
def run_native_components_case():
    n1 = dp.SimNode("n1")
    n2 = dp.SimNode("n2")
    n3 = dp.SimNode("n3")
    n4 = dp.SimNode("n4")
    n5 = dp.SimNode("n5")

    vs = ph1.VoltageSource("VS")
    vs.set_parameters(source_voltage, frequency)

    r_branch = ph1.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph1.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    r_load = ph1.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph1.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph1.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([dp.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, dp.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4, n5],
        [vs, r_branch, l_branch, r_load, l_load, c_load],
    )

    return configure_and_run(system, "DP_Ph1_RLC_StateSpaceExtraction_Native")

## Case 2: native RL branch and DP SSN RLC load

In [6]:
def run_native_branch_ssn_load_case():
    n1 = dp.SimNode("n1")
    n2 = dp.SimNode("n2")
    n3 = dp.SimNode("n3")

    a, b, c, d = create_rlc_load_ssn_matrices()

    vs = ph1.VoltageSource("VS")
    vs.set_parameters(source_voltage, frequency)

    r_branch = ph1.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph1.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    rlc_load = ph1.GenericTwoTerminalVTypeSSN("RLCLoad")
    rlc_load.set_parameters(a, b, c, d)

    vs.connect([dp.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])
    rlc_load.connect([n3, dp.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3],
        [vs, r_branch, l_branch, rlc_load],
    )

    return configure_and_run(system, "DP_Ph1_RLC_StateSpaceExtraction_SSNLoad")

## Case 3: DP SSN RL branch and native RLC load

In [7]:
def run_ssn_branch_native_load_case():
    n1 = dp.SimNode("n1")
    n3 = dp.SimNode("n3")
    n4 = dp.SimNode("n4")
    n5 = dp.SimNode("n5")

    a, b, c, d = create_rl_branch_ssn_matrices()

    vs = ph1.VoltageSource("VS")
    vs.set_parameters(source_voltage, frequency)

    rl_branch = ph1.GenericTwoTerminalVTypeSSN("RLBranch")
    rl_branch.set_parameters(a, b, c, d)

    r_load = ph1.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph1.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph1.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([dp.SimNode.gnd, n1])
    rl_branch.connect([n1, n3])
    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, dp.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n3, n4, n5],
        [vs, rl_branch, r_load, l_load, c_load],
    )

    return configure_and_run(system, "DP_Ph1_RLC_StateSpaceExtraction_SSNBranch")

## Assertions

For this single-phase DP example, the extraction matrix is real-valued and augmented in real/imaginary coordinates.

The expected state count is 6:

- 3 complex extraction states in the compact DP formulation,
- represented as 6 real states in the real-valued augmented matrix.

The finite native-frame DP modes should match the analytical shifted-frequency RLC reference and its conjugates. The redundant trapezoidal/history mode should appear as `z = -1` twice in the real-augmented matrix.

In [8]:
def assert_modal_results(case_name, results):
    state_count = results["state_count"]
    z = results["z"]
    lambdas = results["lambda"]

    assert state_count == 6, f"{case_name}: expected 6 states, got {state_count}"
    assert len(z) == 6, f"{case_name}: expected 6 discrete eigenvalues, got {len(z)}"
    assert (
        len(lambdas) == 6
    ), f"{case_name}: expected 6 continuous eigenvalues, got {len(lambdas)}"

    redundant_modes = np.abs(z + 1.0) < 1e-9
    assert np.count_nonzero(redundant_modes) == 2, (
        f"{case_name}: expected two redundant z=-1 modes, "
        f"got {np.count_nonzero(redundant_modes)}"
    )

    finite_lambdas = lambdas[np.isfinite(lambdas.real) & np.isfinite(lambdas.imag)]
    assert len(finite_lambdas) == 4, (
        f"{case_name}: expected four finite continuous-time modes, "
        f"got {len(finite_lambdas)}"
    )

    np.testing.assert_allclose(
        sort_complex(finite_lambdas),
        sort_complex(lambda_dp_augmented),
        rtol=1e-6,
        atol=1e-6,
        err_msg=f"{case_name}: finite native-frame DP modes do not match",
    )


def assert_same_finite_modes(
    reference_case_name, reference_results, case_name, results
):
    reference_lambdas = reference_results["lambda"]
    lambdas = results["lambda"]

    reference_finite = reference_lambdas[
        np.isfinite(reference_lambdas.real) & np.isfinite(reference_lambdas.imag)
    ]
    finite = lambdas[np.isfinite(lambdas.real) & np.isfinite(lambdas.imag)]

    np.testing.assert_allclose(
        sort_complex(finite),
        sort_complex(reference_finite),
        rtol=1e-6,
        atol=1e-6,
        err_msg=(f"{case_name}: finite modes do not match " f"{reference_case_name}"),
    )

## Run validation

The three model variants are simulated and checked against the same analytical native-frame DP reference.

In [ ]:
cases = {
    "native_components": run_native_components_case(),
    "native_branch_ssn_load": run_native_branch_ssn_load_case(),
    "ssn_branch_native_load": run_ssn_branch_native_load_case(),
}

results_by_case = {}

for case_name, sim in cases.items():
    results = get_modal_results(sim)
    results_by_case[case_name] = results

    print(f"\n{case_name}")
    print("state count:", results["state_count"])

    print("\ndiscrete eigenvalues z:")
    print_complex_vector(results["z"])

    print("\ncontinuous eigenvalues lambda, native DP frame:")
    print_complex_vector(results["lambda"])

    assert_modal_results(case_name, results)

reference_case_name = "native_components"
reference_results = results_by_case[reference_case_name]

for case_name, results in results_by_case.items():
    if case_name == reference_case_name:
        continue
    assert_same_finite_modes(reference_case_name, reference_results, case_name, results)

print("\nAll DP Ph1 state-space extraction and modal-analysis checks passed.")